Goal:

Create tensors (shape, dtype, device)

Do math (broadcasting, indexing, views vs copies, in‑place ops)

Use autograd: build a graph, run .backward(), inspect gradients

(Optional) Jacobian/Hessian & a tiny NN step

We’ll keep it lightweight—no training loops yet.

In [1]:
# pip install torch matplotlib
import torch
import numpy as np
import matplotlib.pyplot as plt
 
print("PyTorch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

PyTorch: 2.9.1+cpu
Device: cpu


Make tensors (and move to device)

In [2]:
# from Python lists / NumPy arrays
t1 = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)           # copies data
t2 = torch.from_numpy(np.array([10, 20, 30], dtype=np.int64))            # shares memory with NumPy
t3 = torch.zeros((2, 3), dtype=torch.float32, device=device)
t4 = torch.randn(3, 4, device=device)                                    # ~N(0,1)
 
print(t1.shape, t1.dtype, t1.device)
print(t2, t2.dtype)
print(t3)
print(t4.mean().item(), t4.std().item())
 
# to/from device
cpu_t4 = t4.to("cpu")
cuda_t1 = t1.to(device)

torch.Size([2, 3]) torch.float32 cpu
tensor([10, 20, 30]) torch.int64
tensor([[0., 0., 0.],
        [0., 0., 0.]])
-0.32502397894859314 0.9164012670516968


Basic ops & broadcasting

In [3]:
a = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
b = torch.tensor([10., 20., 30.])      # shape (3,)
c = a + b                              # broadcast (2,3) + (3,) -> (2,3)
d = a @ torch.tensor([[1.],[0.5],[0.]], dtype=torch.float32)  # matmul -> (2,1)
 
print("c:\n", c)
print("d:\n", d)
 
# elementwise, reduction, transpose
print("a * 2:\n", a * 2)
print("mean over rows:", a.mean(dim=1))
print("transpose:\n", a.T)

c:
 tensor([[11., 22., 33.],
        [14., 25., 36.]])
d:
 tensor([[2.0000],
        [6.5000]])
a * 2:
 tensor([[ 2.,  4.,  6.],
        [ 8., 10., 12.]])
mean over rows: tensor([2., 5.])
transpose:
 tensor([[1., 4.],
        [2., 5.],
        [3., 6.]])


Indexing & views vs copies (reshape/contiguous)

In [4]:
x = torch.arange(12).reshape(3,4)
print("x:\n", x)
 
# slicing gives a VIEW (no copy)
row = x[1]
row[:] = -1
print("after modifying row view:\n", x)
 
# reshape vs view
y = torch.arange(6)
y2 = y.view(2,3)       # view requires contiguous memory
y3 = y.reshape(2,3)    # can copy if needed
y[0] = 999
print("y2 sees change:\n", y2)
 
# concatenation & stacking
cat = torch.cat([torch.ones(2,3), torch.zeros(2,3)], dim=0)  # (4,3)
stk = torch.stack([torch.ones(3), torch.zeros(3)], dim=1)    # (3,2)

x:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
after modifying row view:
 tensor([[ 0,  1,  2,  3],
        [-1, -1, -1, -1],
        [ 8,  9, 10, 11]])
y2 sees change:
 tensor([[999,   1,   2],
        [  3,   4,   5]])


In‑place operations (and autograd caution)

In [5]:
z = torch.ones(3)
z.add_(5)        # in-place (trailing underscore)
print("z:", z)
 
# In-place ops can break autograd if applied to tensors that require grad.
# Prefer out-of-place ops during gradient tracking.

z: tensor([6., 6., 6.])


Autograd 101 — scalar loss .backward()

In [6]:
# requires_grad makes PyTorch track ops to build a graph
x = torch.linspace(-3, 3, steps=7, requires_grad=True)
y = (x**2).sum()          # y is a scalar
print("y:", y.item(), "| grad_fn:", y.grad_fn)
 
y.backward()              # dy/dx
print("x.grad:", x.grad)  # should be 2x
 
# always zero grads before next backward on same leaf tensors
x.grad.zero_()

y: 28.0 | grad_fn: <SumBackward0 object at 0x000001AE18D672E0>
x.grad: tensor([-6., -4., -2.,  0.,  2.,  4.,  6.])


tensor([0., 0., 0., 0., 0., 0., 0.])

Autograd: vector → scalar (mean loss), chain rule

In [7]:
W = torch.randn(3, 2, requires_grad=True)  # weight matrix (3x2)
b = torch.zeros(2, requires_grad=True)
 
X = torch.randn(5, 3)      # 5 samples, 3 features
scores = X @ W + b         # (5,2)
loss = scores.pow(2).mean()# simple MSE-like scalar
 
print("loss grad_fn:", loss.grad_fn)
loss.backward()
print("W.grad shape:", W.grad.shape, "| b.grad:", b.grad)
 
# reset grads before reusing the same parameters
W.grad.zero_(); b.grad.zero_()

loss grad_fn: <MeanBackward0 object at 0x000001AE18E6E140>
W.grad shape: torch.Size([3, 2]) | b.grad: tensor([-0.4939,  0.0235])


tensor([0., 0.])

Turn off grad tracking

In [8]:
with torch.no_grad():
    W += 0.1  # e.g., manual update step without tracking
 
# or detach a tensor from the graph
detached = scores.detach()
assert not detached.requires_grad

Visual cues: graph “string” & grad flow

In [9]:
# Show grad functions involved (simple peek)
x = torch.randn(4, requires_grad=True)
y = x * 3 + 2
z = (y.sigmoid()).sum()
print("z grad_fn chain:", z.grad_fn, " <- ", z.grad_fn.next_functions[0][0])
 
z.backward()
print("x.grad:", x.grad)

z grad_fn chain: <SumBackward0 object at 0x000001AE18AA1ED0>  <-  <SigmoidBackward0 object at 0x000001AE18E90D30>
x.grad: tensor([0.2469, 0.5134, 0.6609, 0.4884])


# pip install torchviz
# from torchviz import make_dot
# make_dot(z, params={"x": x}).render("autograd_graph", format="png")

In [10]:
!pip install torchviz

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [graphviz]
   ---------------------------------------- 0/2 [graphviz]
   ---------------------------------------- 0/2 [graphviz]
   ---------------------------------------- 0/2 [graphviz]
   ---------------------------------------- 0/2 [graphviz]
   -------------------- ------------------- 1/2 [torchviz]
   ---------------------------------------- 2/2 [torchviz]




[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from torchviz import make_dot
make_dot(z, params={"x": x}).render("autograd_graph", format="png")

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH